In [ ]:
from ultralytics import YOLO
model = YOLO("yolo11n.pt")
model.train(
    data="data-train_Pilot-Largest.yaml",   # file YAML cấu hình dữ liệu
    epochs=100,
    imgsz=640,
    batch=16,
    name="yolo11_vesseldetectionPilot_L_train1_group5",
    device=0 , # GPU 0. Đổi thành 'cpu' nếu không có GPU
    workers=0,
    degrees=10.0,                 # Xoay ảnh ngẫu nhiên ±10 độ
    translate=0.1,                # Dịch ảnh ±10%
    scale=0.5,                    # Phóng to/thu nhỏ ảnh ±50%
    shear=10.0,                   # Nghiêng ảnh ±10 độ
    perspective=0.0005,           # Biến dạng phối cảnh
    flipud=0.5,                   # Lật dọc ảnh với xác suất 0.5
    #fliplr=0.5,                   # Lật ngang ảnh với xác suất 0.5
    mosaic=0,                   # Sử dụng mosaic augmentation (4 ảnh trong 1)
    mixup=0,                    # Sử dụng mixup với xác suất 0.2
    hsv_h=0.015,                  # Thay đổi hue
    hsv_s=0.7,                    # Thay đổi saturation
    hsv_v=0.4   
)

In [ ]:
metrics = model.val(
    data="data.yaml",  # Đường dẫn tới file YAML định nghĩa tập test
    split='test',      # Đảm bảo sử dụng tập test, nếu có trong YAML
    imgsz=640,
    batch=16,
    device=0,
    workers=0          # Hoặc 'cpu' nếu không dùng GPU
)

# In kết quả đánh giá
print(metrics)

In [ ]:
from ultralytics import YOLO

 #Load model đã huấn luyện (tên nằm trong folder runs/detect/)
model = YOLO("/home/aiplatform/projects/Yolo11/runs/detect/yolo11_vesseldetectionPilot_L_train1_group5/weights/best.pt")

# Đánh giá trên tập test được khai báo trong data.yaml (trường 'test')
metrics = model.val(
    data="data.yaml",  # Đường dẫn tới file YAML định nghĩa tập test
    split='test',      # Đảm bảo sử dụng tập test, nếu có trong YAML
    imgsz=640,
    batch=16,
    device=0,
    workers=0          # Hoặc 'cpu' nếu không dùng GPU
)

# In kết quả đánh giá
print(metrics)

In [ ]:
result = model.predict(
    source="/home/aiplatform/projects/Yolo11/test/images/img_cameras_2023-06-27-14-04-13_14_jpg.rf.e8cdd6ef8fcda3e193156494434a6d48.jpg",
    show=True,     # Hiển thị ảnh có bounding boxes
    save=True      # Lưu ảnh kết quả vào runs/
)


In [13]:
import os
from collections import defaultdict

# Cấu hình
#labels_dir ='/home/aiplatform/projects/SAM/test_combined_outputs/part4_YL/labels'
#image_dir = '/home/aiplatform/projects/SAM/test_combined_outputs/part4_YL/images'
labels_dir ='/home/aiplatform/projects/Yolo11/train_Flux/labels'
image_dir = '/home/aiplatform/projects/Yolo11/train_Flux/images'

# Khởi tạo bộ đếm
total_images = 0
total_objects = 0
class_counts = defaultdict(int)

# Đếm số file ảnh
total_images = len([f for f in os.listdir(image_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])

# Duyệt qua từng file .txt trong labels/
for label_file in os.listdir(labels_dir):
    if not label_file.endswith('.txt'):
        continue

    with open(os.path.join(labels_dir, label_file), 'r') as f:
        lines = f.readlines()
        total_objects += len(lines)
        for line in lines:
            if line.strip() == "":
                continue
            class_id = line.strip().split()[0]
            class_counts[class_id] += 1

# In kết quả
print(f"Số file ảnh       : {total_images}")
print(f"Tổng số objects   : {total_objects}")
print(f"Số object theo class_id:")
for cls_id, count in sorted(class_counts.items(), key=lambda x: int(x[0])):
    print(f"  Class {cls_id}: {count}")


Số file ảnh       : 7910
Tổng số objects   : 17017
Số object theo class_id:
  Class 0: 425
  Class 1: 295
  Class 2: 7781
  Class 3: 2851
  Class 4: 773
  Class 5: 4892


In [12]:
import os
import shutil

# Thư mục nguồn và đích

src_dir = "/home/aiplatform/projects/Yolo11/train/images"

dst_dir =  "/home/aiplatform/projects/Yolo11/train_Flux/images"

# Tạo thư mục đích nếu chưa tồn tại
os.makedirs(dst_dir, exist_ok=True)

# Duyệt và copy từng file
for file_name in os.listdir(src_dir):
    src_file = os.path.join(src_dir, file_name)
    dst_file = os.path.join(dst_dir, file_name)
    if os.path.isfile(src_file):  # Bỏ qua thư mục
        shutil.copy(src_file, dst_file)
        #print(f"Đã copy: {file_name}")
print(f"Đã copy xong")

Đã copy xong


In [6]:
import os
import shutil

# Thư mục nguồn và đích
src_dir = "/home/aiplatform/projects/test/data/VESSELimg/Fluxgenerated1_merge/train1_Flux_group3_YL/images"
dst_dir =  "/home/aiplatform/projects/Yolo11/train_Flux/images"

# Tạo thư mục đích nếu chưa tồn tại
os.makedirs(dst_dir, exist_ok=True)

# Duyệt và copy từng file nếu chưa tồn tại ở đích
for file_name in os.listdir(src_dir):
    src_file = os.path.join(src_dir, file_name)
    dst_file = os.path.join(dst_dir, file_name)
    if os.path.isfile(src_file):
        if not os.path.exists(dst_file):  # Kiểm tra file đã tồn tại chưa
            shutil.copy(src_file, dst_file)
            # print(f"Đã copy: {file_name}")
        # else:
            # print(f"Bỏ qua (đã tồn tại): {file_name}")

print("Đã copy xong")


Đã copy xong


In [ ]:
import os
import shutil
import xml.etree.ElementTree as ET

# Cấu hình đường dẫn
data_dir = r"/home/aiplatform/projects/SAM/test_combined_outputs/part5"
out_dir =r"/home/aiplatform/projects/SAM/test_combined_outputs/part5_YL"
images_dir = os.path.join(out_dir, 'images')   # ảnh đầu ra
labels_dir = os.path.join(out_dir, 'labels')   # nhãn YOLO đầu ra
os.makedirs(images_dir, exist_ok=True)
os.makedirs(labels_dir, exist_ok=True)

# Danh sách lớp cố định (YOLO yêu cầu theo chỉ số)
CLASSES = ['Buoy', 'Chemical', 'Container', 'Passenger-RoRo', 'Pilot', 'Tugboat']

# Hàm chuyển bbox VOC -> YOLO
def convert_bbox(size, box):
    dw = 1.0 / size[0]
    dh = 1.0 / size[1]
    x_center = (box[0] + box[2]) / 2.0
    y_center = (box[1] + box[3]) / 2.0
    width = box[2] - box[0]
    height = box[3] - box[1]
    return x_center * dw, y_center * dh, width * dw, height * dh

# Duyệt toàn bộ file XML trong thư mục gốc
for file in os.listdir(data_dir):
    if not file.endswith('.xml'):
        continue

    xml_path = os.path.join(data_dir, file)
    tree = ET.parse(xml_path)
    root = tree.getroot()

    # Dùng tên file ảnh trùng với tên file xml (đổi đuôi sang .jpg)
    filename = os.path.splitext(file)[0] + '.png'
    img_src_path = os.path.join(data_dir, filename)
    img_dst_path = os.path.join(images_dir, filename)

    # Copy ảnh sang thư mục images/
    if os.path.exists(img_src_path):
        shutil.copy(img_src_path, img_dst_path)
    else:
        print(f"Ảnh không tồn tại: {img_src_path}")
        continue

    # Kích thước ảnh
    size = root.find('size')
    w = int(size.find('width').text)
    h = int(size.find('height').text)

    # Tạo file nhãn YOLO
    label_path = os.path.join(labels_dir, file.replace('.xml', '.txt'))
    with open(label_path, 'w') as out:
        for obj in root.findall('object'):
            cls_name = obj.find('name').text
            if cls_name not in CLASSES:
                continue
            cls_id = CLASSES.index(cls_name)
            xml_box = obj.find('bndbox')
            bbox = (
                int(xml_box.find('xmin').text),
                int(xml_box.find('ymin').text),
                int(xml_box.find('xmax').text),
                int(xml_box.find('ymax').text)
            )
            yolo_box = convert_bbox((w, h), bbox)
            out.write(f"{cls_id} {' '.join(f'{x:.6f}' for x in yolo_box)}\n")


In [14]:
import os
import zipfile

def zip_folder(folder_path, output_zip_path):
    with zipfile.ZipFile(output_zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, folder_path)
                zipf.write(file_path, arcname)
    print(f"Đã nén thư mục '{folder_path}' thành '{output_zip_path}'")

# Ví dụ sử dụng
folder_to_zip = '/home/aiplatform/projects/Yolo11/train_Flux/'         # Thay bằng đường dẫn tới thư mục cần nén
output_zip = '/home/aiplatform/projects/Yolo11/train_Flux.zip'        # Thay bằng đường dẫn file zip đầu ra

zip_folder(folder_to_zip, output_zip)


Đã nén thư mục '/home/aiplatform/projects/Yolo11/train_Flux/' thành '/home/aiplatform/projects/Yolo11/train_Flux.zip'
